[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/02_pd_orientation.ipynb)

# R2-Q1 Week 1 — Orient on PlantDoc

This notebook orients you on the PlantDoc dataset: 27 classes (17 disease and 10 healthy) across 13 plant species, captured under field conditions that contrast with PlantVillage's controlled lab setup. Together with `01_pv_orientation.ipynb`, the inspection here supports your Week 1 EQ#1 report and the written pre-commits on aggregation level, class-space remapping, and statistical test that R2-Q1 requires before any training begins.

By the end of this notebook you will have:

- A PlantDoc metadata table (`pd_metadata.parquet`) keyed by image path, with class label and host species per image, ready for the transfer evaluation in Week 3.
- Summary statistics on the PlantDoc class space — total image count, per-class distribution, host-species coverage, disease-vs-healthy class split — saved as `pd_class_summary.parquet`.
- A `pd_orientation_summary.json` recording dataset path, image and class counts, host-species count, and key distributional facts, ready as Week 1 EQ#1 input.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R2-Q1 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load the PlantDoc metadata

PlantDoc (Singh et al. 2020) is the second of the two datasets you'll
use in R2-Q1. Where PlantVillage is roughly 54,000 lab-condition leaf
images shot against uniform backgrounds, PlantDoc is roughly 2,500
*field-condition* images — photos of diseased and healthy leaves in
real-world settings, heterogeneous in resolution, lighting, and
background. That heterogeneity is exactly the point: PlantDoc is the
canonical test bed for measuring whether a PV-trained classifier
transfers to the messy conditions a real deployment would see.

The function below returns two things, not one — a `(metadata_df,
hf_dataset)` tuple. The DataFrame is your handle on what's in the
dataset; the `hf_dataset` is a Hugging Face Dataset object that holds
the image bytes themselves. Both are needed because the PyTorch wrapper
you'll use next week (`PlantDocDataset`) takes both: the DataFrame tells
it which rows to iterate over, the Dataset object holds the actual
images.

The first time you call this in a fresh session, expect a 30-to-90-
second pause while the ~950 MB dataset gets fetched from the Hugging
Face Hub. Subsequent calls in the same session are instant.

In [ ]:
### 2.1) Load PlantDoc data
metadata, hf_dataset = iri.load_plantdoc()

print(f"Metadata shape:  {metadata.shape[0]:,} rows × {metadata.shape[1]} columns")
print(f"Columns:         {list(metadata.columns)}")
print(f"HF Dataset:      {len(hf_dataset):,} rows")
print()
print("First five rows:")
metadata.head()

Seven columns of metadata, all string or simple types. Notice what's
*not* here: there's no `image_path` column. PlantDoc's images don't
live as files in a directory you can point to — they live inside the
`hf_dataset` object as PIL Images, indexed by pandas row position.

Two columns carry the same information in different forms. `class_label`
is the upstream folder name verbatim from Singh et al. 2020 ("Apple
Scab Leaf", "Tomato leaf bacterial spot", etc.); `class_idx` is an
integer (0 to 27) assigned by sorting class labels alphabetically with
case sensitivity. The integer form is what your classifier will
predict.

The remaining five columns:

- `host` — normalized host name (e.g., "Apple", "Tomato"). Hand-curated
  from the irregular upstream folder names.
- `disease` — lowercased disease name, or `"healthy"` for healthy leaves.
- `is_healthy` — boolean shorthand for `disease == "healthy"`.
- `split` — `"train"` or `"test"`, from the upstream partition.
- `filename` — original filename verbatim, including any URL-encoded
  characters and double extensions.

**Looking up an image.** Each row's pandas index is the position to use
in `hf_dataset`. To get the image for the first row:

    image = hf_dataset[0]["image"]    # a PIL Image

You'll see this pattern again in Section 6 when you display sample
images.

## 3) Inspect class structure

PlantDoc has 28 classes — 27 disease-on-host classes plus one orphan
you need to know about. Unlike PlantVillage's tidy `host___disease`
naming convention, PlantDoc's class names are inconsistent in
capitalization, word order, and where the word "leaf" appears. The
`host` and `disease` columns are a hand-curated normalization layer
that smooths this out; the `class_label` column preserves the upstream
string verbatim because the link back to Singh et al. 2020 should be
exact.

Start by looking at one row in detail.

In [ ]:
### 3.1) Walk through one row
example = metadata.iloc[0]
print(f"class_label:  {example['class_label']}")
print(f"class_idx:    {example['class_idx']}")
print(f"host:         {example['host']}")
print(f"disease:      {example['disease']}")
print(f"is_healthy:   {example['is_healthy']}")
print(f"split:        {example['split']}")
print(f"filename:     {example['filename']}")

The `class_label` string is what the upstream PlantDoc repository
named the folder this image came from. A few examples of how
inconsistent these strings are:

- `Apple Scab Leaf` — Title Case, disease before "Leaf"
- `Tomato Septoria leaf spot` — mixed case, "leaf spot" as a compound
- `grape leaf` — all lowercase
- `Tomato leaf bacterial spot` — host first, then "leaf", then disease
- `Soyabean` — upstream misspelling of "Soybean", preserved verbatim

This kind of irregularity is normal for web-scraped data; the
discipline of preserving upstream strings is what lets you trace any
individual image back to Singh et al. 2020 unambiguously. Use the
`host` and `disease` columns for any analysis grouping — those are
normalized.

One class is unusual enough to surface explicitly. The class
"Tomato two spotted spider mites leaf" has 2 images in the training
split and 0 images in the test split — a real artifact of the upstream
dataset, not a bug. Singh et al. 2020 and downstream benchmarks like
Ahmad et al. 2023 drop this class implicitly when they report "27
classes." This dataset preserves it for upstream fidelity. You'll need
to decide whether to drop it from your own evaluation; that choice
becomes part of your EQ#1 pre-commit in Section 8.

In [ ]:
### 3.2) Count classes and per-class images
class_counts = metadata['class_label'].value_counts()

print(f"Number of classes:  {len(class_counts)}")
print()
print(f"Largest class:      {class_counts.idxmax()}")
print(f"                    {class_counts.max():,} images")
print(f"Smallest class:     {class_counts.idxmin()}")
print(f"                    {class_counts.min():,} images")
print(f"Median count:       {int(class_counts.median()):,} images per class")
print(f"Ratio largest:smallest:  {class_counts.max() / class_counts.min():.1f}×")
print()

# Surface the orphan class explicitly: which classes have 0 test images?
split_counts = metadata.groupby(['class_label', 'split']).size().unstack(fill_value=0)
no_test = split_counts[split_counts.get('test', 0) == 0]
print("Classes with 0 test images:")
print(no_test if len(no_test) > 0 else "  (none)")

Two things to take in here.

The ratio between largest and smallest classes is well above the noise
floor — PlantDoc isn't balanced. When you train next week on
PlantVillage and evaluate on PlantDoc, the per-class accuracy you'd
compute on the smallest PD classes will come from very few test images
(typically 8 to 12 per class, sometimes fewer). This is the noise
problem the R2-Q1 Considerations section flagged: per-class accuracy
on PD test sets is too noisy to support reliable claims at the per-
class level.

The aggregation question this raises is one of the three pre-commits
you'll write in Section 8. The decision isn't whether to aggregate
(you essentially have to) but at what level — host group, disease
category, or some other grouping you can defend.

The orphan class you saw flagged above is also a per-class-count issue,
just an extreme one. Zero test images means you can't measure your
classifier's accuracy on that class no matter what. Dropping it from
the evaluation is the standard move; preserving it in the metadata for
upstream fidelity is what this dataset does.

In [ ]:
### 3.3) Plot per-class image counts by split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
split_counts_sorted = (
    split_counts
    .assign(total=lambda d: d[["train", "test"]].sum(axis=1))
    .sort_values("total", ascending=False)   # highest totals at top
)

# Convert from wide to long format for seaborn
plot_df = (
    split_counts_sorted[["train", "test"]]
    .reset_index()
    .melt(
        id_vars=split_counts_sorted.index.name or "index",
        value_vars=["train", "test"],
        var_name="split",
        value_name="count"
    )
)

# Rename class column cleanly
plot_df = plot_df.rename(columns={split_counts_sorted.index.name or "index": "class"})

# Preserve descending class order
class_order = split_counts_sorted.index.tolist()

# -----------------------------
# Pastel-ish colors
# -----------------------------
palette = {
    "train": "#9ecae1",  # soft blue
    "test": "#fdae6b",   # soft orange
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 10))

left_values = pd.Series(0, index=class_order)

for split in ["train", "test"]:
    split_df = (
        plot_df[plot_df["split"] == split]
        .set_index("class")
        .loc[class_order]
    )

    ax.barh(
        y=class_order,
        width=split_df["count"],
        left=left_values,
        label=split,
        color=palette[split],
        alpha=0.85
    )

    left_values += split_df["count"]

ax.set_xlabel("Number of images")
ax.set_ylabel("")
ax.set_title("Images per class in PlantDoc: train + test")

# Put highest values at the top
ax.invert_yaxis()

# Cleaner legend placement
ax.legend(
    title="Split",
    loc="center right",
    bbox_to_anchor=(0.98, 0.82),
    frameon=True
)

sns.despine(left=True, bottom=False)

plt.tight_layout()
plt.show()

## 4) Inspect host species

"Host" here means the plant species the leaf came from. Aggregating
from classes to hosts collapses across diseases for the same plant —
every disease of tomato, plus healthy tomato, becomes one "Tomato"
group. This is one candidate aggregation level for your transfer
evaluation: rather than asking "how does my classifier do on tomato
leaf bacterial spot?" you ask "how does it do on tomatoes overall?"

In [ ]:
### 4.1) Count hosts and images per host
host_counts = metadata['host'].value_counts()

print(f"Number of host species:  {len(host_counts)}")
print()
print("Images per host:")
print(host_counts.to_string())

Thirteen distinct hosts in PlantDoc, versus PlantVillage's fourteen.
Tomato is the most-represented host in both datasets, but the absolute
counts differ by an order of magnitude — PlantDoc's per-host counts
are in the dozens-to-hundreds range, not the thousands-to-tens-of-
thousands range.

Two upstream conventions are worth noting because they'll matter when
you build your PV→PD remapping in Section 8:

- "Corn" in PlantDoc corresponds to "Corn_(maize)" in PlantVillage —
  same plant, different upstream label string.
- "Soyabean" in PlantDoc is the upstream misspelling of "Soybean",
  which is how PlantVillage spells it.

If you're aligning host names across the two datasets, you'll need to
handle these by hand. Whichever direction you go (rename one to match
the other, or use a translation dict in your remapping), document it
explicitly — a reader checking your work shouldn't have to guess.

In [ ]:
### 4.2) Plot images per host
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
plot_df = (
    host_counts
    .sort_values(ascending=False)
    .reset_index()
)

plot_df.columns = ["host_species", "count"]

# -----------------------------
# Categorize counts for PlantDoc scale
# -----------------------------
def count_category(n):
    if n > 1000:
        return "> 1000"
    elif n > 500:
        return "501–1000"
    elif n > 250:
        return "251–500"
    else:
        return "≤ 250"

plot_df["count_group"] = plot_df["count"].apply(count_category)

group_order = ["> 1000", "501–1000", "251–500", "≤ 250"]

palette = {
    "> 1000": "#9ecae1",     # soft blue
    "501–1000": "#a1d99b",  # soft green
    "251–500": "#fdae6b",   # soft orange
    "≤ 250": "#fbb4b9",     # soft pink
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=plot_df,
    x="host_species",
    y="count",
    hue="count_group",
    hue_order=group_order,
    palette=palette,
    dodge=False,
    order=plot_df["host_species"],
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Number of images")
ax.set_title("Images per host species in PlantDoc")

plt.xticks(rotation=45, ha="right")

ax.legend(
    title="Image count",
    loc="upper right",
    frameon=True
)

sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

## 5) Inspect disease types

The `disease` column carries a different cut through the same data.
Where the class label tells you "tomato with bacterial spot" and the
host column tells you "tomato," the disease column tells you "bacterial
spot" regardless of host.

Two PD-specific conventions to know about. First, disease names are
*lowercased* in the `disease` column even when the upstream
`class_label` had Title Case (`Apple Scab Leaf` becomes
`disease = "scab"`). This differs from how PlantVillage's loader
handles disease names — be deliberate about case-insensitive matching
if you're comparing across datasets. Second, the disease "two spotted
spider mites" is technically a pest, not a pathological disease; it
shows up in PlantDoc because the upstream folder named it as a class.

Start with the simplest cut: healthy versus diseased.

In [ ]:
### 5.1) Healthy vs diseased
n_healthy = metadata['is_healthy'].sum()
n_diseased = (~metadata['is_healthy']).sum()
print(f"Healthy images:   {n_healthy:,}  ({n_healthy / len(metadata):.1%})")
print(f"Diseased images:  {n_diseased:,}  ({n_diseased / len(metadata):.1%})")

Now look at how many distinct disease labels there are, and what they
are.

In [ ]:
### 5.2) Count and list distinct diseases
unique_diseases = sorted(metadata['disease'].unique())

print(f"Number of distinct disease labels:  {len(unique_diseases)}")
print()
print("All disease labels:")
for d in unique_diseases:
    print(f"  {d}")

The R2-Q1 Considerations section flagged "fungal, bacterial, or viral
disease categories" as one of the candidate aggregation levels. To use
that grouping, you need a mapping from each disease label to its broad
category.

PlantDoc doesn't provide this mapping. The cell below provides a
starter mapping covering the disease labels in the dataset, based on
common plant-pathology references. **Treat it as a draft, not as
authoritative.** Each entry is a verifiable claim ("scab on apple is
fungal") that you should check against your reading list before
locking it in.

Two PD-specific subtleties. First, this mapping uses *lowercased* keys
because that's how PD's `disease` column reads — if you build a
parallel mapping for PlantVillage later, you'll need to handle case
carefully. Second, the "two spotted spider mites" entry is a pest,
not a disease in the pathological sense — same edge case PlantVillage
had with the same pest, and the same choice you'll need to make about
how to group or drop it.

In [ ]:
### 5.3) Apply a starter category mapping
# Starter mapping from PD disease label to broad category.
# Categories: fungal, bacterial, viral, pest, healthy.
# Verify each entry against your reading list before using this in
# your EQ#1 pre-commit. Disease names below match what
# `iri.load_plantdoc()` produces in the `disease` column (lowercased).
pd_disease_category = {
    "scab":                      "fungal",
    "rust":                      "fungal",
    "black rot":                 "fungal",
    "early blight":              "fungal",
    "late blight":               "fungal",
    "leaf blight":               "fungal",
    "gray leaf spot":            "fungal",
    "septoria leaf spot":        "fungal",
    "powdery mildew":            "fungal",
    "leaf mold":                 "fungal",
    "mold":                      "fungal",
    "bacterial spot":            "bacterial",
    "mosaic virus":              "viral",
    "yellow leaf curl virus":    "viral",
    "two spotted spider mites":  "pest",
    "healthy":                   "healthy",
}

# Apply the mapping. Any disease label not in the dict shows up as
# "UNMAPPED" — if you see that, your mapping is incomplete and you
# need to extend it before relying on it for evaluation.
pd_categorized = metadata['disease'].map(pd_disease_category).fillna("UNMAPPED")

print("Images per category:")
print(pd_categorized.value_counts().to_string())
print()
unmapped_count = (pd_categorized == "UNMAPPED").sum()
if unmapped_count > 0:
    unmapped_diseases = metadata.loc[pd_categorized == "UNMAPPED", 'disease'].unique()
    print(f"WARNING: {unmapped_count} images have unmapped disease labels:")
    for d in unmapped_diseases:
        print(f"  {d}")

The category counts are imbalanced in much the same direction as
PlantVillage's: fungal diseases dominate, bacterial and viral are
minority categories. PlantDoc's "pest" category is the same single-
disease edge case PV has — spider mites on tomato.

Two things this surfaces for your evaluation work.

Aggregating to the three-category level (fungal / bacterial / viral)
gives you a small number of categories with workable image counts per
category. This is the "broad disease category" aggregation level the
R2-Q1 page mentions.

The category counts here will be the *upper bound* on your test-set
size at this aggregation level once you've matched against
PlantVillage's class space (Section 7 sets up that comparison; Section
8 commits to a remapping).

## 6) Look at sample images

You've inspected the metadata. Now look at the actual images, because
the visual difference between PlantDoc and PlantVillage is what makes
the lab-to-field gap exist in the first place. PlantVillage was shot
in a studio; PlantDoc was scraped from the open web. Below, four hosts
side by side, healthy and diseased, displayed *at their original
resolution* — no resizing before display. The heterogeneity is the
point.

The image display uses the lookup pattern from Section 2: each row's
pandas index is the position to read from `hf_dataset`. The defensive
`.convert("RGB")` call (which `PlantDocDataset` does for you when you
use it later) is reproduced here because PlantDoc includes at least
one CMYK image, and silently passing CMYK to matplotlib is the kind of
bug that's hard to debug if it happens far from where you can see it.

In [ ]:
### 6.1) Display sample images at original resolution
# Four hosts chosen to overlap with the PV orientation notebook's
# choices. Each contributes one healthy and one diseased sample where
# available. Image titles include the actual resolution to make the
# heterogeneity visible.
sample_hosts = ['Apple', 'Grape', 'Potato', 'Tomato']

samples = []
for host in sample_hosts:
    for state, mask in [('healthy', metadata['is_healthy']),
                        ('diseased', ~metadata['is_healthy'])]:
        host_rows = metadata[(metadata['host'] == host) & mask]
        if len(host_rows) == 0:
            # PlantDoc doesn't have every host in both states.
            # Print a note instead of silently skipping.
            print(f"No {state} samples for host {host} — skipping.")
            continue
        chosen = host_rows.sample(n=1, random_state=42).iloc[0]
        # Use the pandas row index to look up the image in hf_dataset.
        image = hf_dataset[int(chosen.name)]["image"].convert("RGB")
        samples.append((chosen, image))

# Display in a 2-column grid, one row per host-state combination.
# Each axis is sized generously so the native aspect ratio is visible.
n = len(samples)
fig, axes = plt.subplots(n // 2, 2, figsize=(12, 5 * (n // 2)))
for ax, (row, image) in zip(axes.flat, samples):
    ax.imshow(image)
    w, h = image.size
    state_label = "healthy" if row['is_healthy'] else "diseased"
    ax.set_title(
        f"{row['host']} — {state_label} ({row['disease']})\n{w}×{h} pixels",
        fontsize=10
    )
    ax.axis('off')
plt.tight_layout()
plt.show()

A few things to take in.

**Backgrounds vary.** Some images are shot against soil, some against
sky, some against other foliage, some on a human hand or a table.
There is no convention. A model trained on PlantVillage's uniform
gray backdrop has nothing in its training history that helps it
recognize "this is a leaf against soil" as the same kind of input as
"this is a leaf against gray cardboard."

**Lighting varies.** Some images are overexposed, some are in deep
shade, some have dappled light, some have direct flash. Compare this
to PV's controlled studio lighting where exposure and shadow are
roughly the same image to image.

**Resolution varies.** The image titles above show wildly different
pixel counts — width ranges from under 200 pixels to over 4,000 pixels
across the full dataset, and aspect ratios are anything but uniform.
When you train your classifier next week, you'll resize all PD images
to whatever input resolution your CNN expects. That resize is itself a
transformation PV images never had to undergo — they were already
roughly uniform — and it's worth thinking about whether the resize is
doing something to the high-resolution PD images that the low-
resolution PV images escape.

**Capture devices vary.** PV used a small number of cameras under
controlled conditions; PD images came from whatever phone, camera, or
microscope the original photographer used. JPEG compression artifacts,
sensor noise patterns, and color profiles all differ across images.
That's part of what your classifier has to handle in transfer.

None of this is a problem for PlantDoc as a dataset — it's exactly
what makes it useful as a *field-condition* reference. The PV → PD gap
exists because a model trained only on uniform-condition photographs
has no signal in its training history for how to ignore the visual
variation that PD introduces. Sections 7 and 8 set up the methodology
pre-commits you'll need before measuring that gap rigorously.

## 7) PV vs PD: what aligns, what doesn't

You've now seen PlantDoc on its own terms. The transfer question R2-Q1
asks (does a PV-trained classifier work on PD images?) requires you to
align the two datasets' class spaces before you can compute anything
meaningful. PV has 38 classes; PD has 28; the *intersection by class
name* is essentially zero because the two datasets name their classes
differently.

What you need to find is the overlap at *coarser* levels — which
hosts appear in both, which diseases appear in both, which host-
disease combinations appear in both. Section 8 turns this overlap
into a concrete class-space remapping you commit to before any
training happens.

Load PV's metadata so you can compare side by side. If you ran Notebook
01 in this session (or a prior session with the cache mounted), the PV
metadata is already cached and the call below is near-instant. If not,
expect a 30-to-60-second download.

In [ ]:
### 7.1) Load PV metadata for comparison
pv_metadata, pv_hf_dataset = iri.load_plantvillage()
print(f"PV metadata:  {len(pv_metadata):,} rows, {pv_metadata['class_label'].nunique()} classes")
print(f"PD metadata:  {len(metadata):,} rows, {metadata['class_label'].nunique()} classes")

In [ ]:
### 7.2) Compare host overlap
pv_hosts = set(pv_metadata['host'].unique())
pd_hosts = set(metadata['host'].unique())

print("PV-only hosts:")
for h in sorted(pv_hosts - pd_hosts):
    print(f"  {h}")
print()
print("PD-only hosts:")
for h in sorted(pd_hosts - pv_hosts):
    print(f"  {h}")
print()
print("Hosts in both (by exact string match):")
for h in sorted(pv_hosts & pd_hosts):
    print(f"  {h}")

Look at the "Hosts in both" list first. Those are the hosts where a
direct PV → PD transfer evaluation makes sense without any host-level
remapping. Hosts in only one dataset are the ones you'll have to
either drop from your evaluation or remap via a translation step
(e.g., does PV's "Corn_(maize)" count as the same host as PD's
"Corn"? Yes for any sensible analysis, but you have to say so in
writing).

This is the first concrete version of the remapping problem. The PV →
PD direct overlap is partial, and you have to commit to how you'll
handle the partial overlap before you train.

In [ ]:
### 7.3) Compare disease overlap (case-insensitive)
# Lowercase both for fair comparison — PV preserves Title Case, PD
# lowercases. The comparison should look at the underlying disease
# regardless of how each dataset capitalized it.
pv_diseases = set(pv_metadata['disease'].str.lower().unique())
pd_diseases = set(metadata['disease'].str.lower().unique())

print(f"PV diseases:  {len(pv_diseases)}")
print(f"PD diseases:  {len(pd_diseases)}")
print()
print("Diseases in both (lowercased, exact string match):")
for d in sorted(pv_diseases & pd_diseases):
    print(f"  {d}")
print()
print("PV-only diseases:")
for d in sorted(pv_diseases - pd_diseases):
    print(f"  {d}")
print()
print("PD-only diseases:")
for d in sorted(pd_diseases - pv_diseases):
    print(f"  {d}")

The exact-match overlap on disease names is smaller than you might
expect. Several diseases have *almost* the same name across the two
datasets but not exactly — even with lowercase normalization, you'll
see PV's "northern leaf blight" and PD's "leaf blight" as distinct
strings when they're plausibly the same disease.

This is the second concrete version of the remapping problem. The
disease-name fields in the two datasets are normalized differently
enough that string equality isn't a reliable test of "same disease."
You'll need to make explicit judgment calls — does PV's "Cedar apple
rust" map to PD's "rust"? does PV's "Leaf scorch" map to anything in
PD? — and write those calls into your EQ#1 pre-commit.

## 8) Lock in your Week 1 decisions and save the file

You've spent this week orienting on PlantVillage (Notebook 01) and
PlantDoc (Sections 2-7 above). You've seen what's in both datasets,
how their class spaces differ, and what kinds of comparisons are
going to be possible and not possible in Week 3.

Now you commit to three decisions, in writing, before Week 2 starts:

1. **Aggregation level.** At what level will you compare PV-internal
   accuracy to PV → PD accuracy?
2. **Class-space remapping.** How will you map between PV's 38
   classes and PD's class space?
3. **Statistical test.** How will you decide whether the two
   accuracy numbers are statistically distinguishable?

The R2-Q1 question page's Considerations section explains why each
of these has to be locked in *before* you run anything: if you pick
your aggregation level after seeing your results, you'll end up
choosing the level that tells the story you want — and your claim
about the lab-to-field gap stops being a real test.

The next three subsections each capture one decision. The cell
patterns are the same: a markdown cell explains what you're
committing to and what your options are, and a code cell adds your
choice to a Python dictionary called `precommit`. You'll be editing
the `choice` and `reasoning` fields in each code cell — those are
where your decisions live.

After all three are recorded, Section 8.4 writes the dictionary to
a file called `precommit.json` in your R2-Q1 output directory.
Section 8.5 reads the file back to confirm the save worked.

`precommit.json` is your **EQ#1 deliverable**. Submit it (plus your
EQ#1 writeup, which draws on the `reasoning` fields and whatever
additional framing the deliverables page asks for) by the end of
Week 1.

### 8.1) Pre-commit your aggregation level

PlantDoc has roughly 2,500 images across 27 classes, which leaves
8-12 test images per class. That's too few for reliable per-class
accuracy claims — the noise on a count that small will swamp any
real signal. So you have to aggregate to a coarser level before you
compare.

The R2-Q1 Considerations section flags two candidate aggregations,
plus the door for a third:

- **Disease category** — fungal, bacterial, viral, plus pest and
  healthy as separate buckets. Roughly 4-5 categories. This is the
  conceptually clean choice: the transfer question is "does the
  model recognize *this is a fungal disease* regardless of host,"
  and disease category captures that directly.
- **Host species group** — apple, tomato, corn, and so on. Roughly
  13 groups in PV. Useful if you care about which host species
  transfer reliably across the lab/field shift. Less directly
  aligned with the transfer question.
- **Another grouping you can defend** — for example, you could
  define your own buckets (e.g., "leaf-attended diseases" vs.
  "fruit-attended diseases"). Higher bar to defend, but possible if
  you have a specific hypothesis.

Pick one and write your reasoning into the code cell below. The
reasoning field is where your EQ#1 writeup gets its content for
this decision — be specific about *why* you picked what you picked.

In [ ]:
### 8.1) Record aggregation level
precommit = {}

precommit["aggregation_level"] = {
    "choice": "disease_category",  # edit to your choice (or write your own)
    "reasoning": (
        # Edit this string. Defend your choice in 2-4 sentences.
        # The text you write here becomes the source material for your
        # EQ#1 writeup on this decision.
        "Disease category is the conceptually meaningful unit for the "
        "transfer question. The R2-Q1 hypothesis is about whether a "
        "lab-trained model recognizes diseases (not host species), and "
        "aggregating to fungal/bacterial/viral/pest/healthy captures that "
        "directly. PlantDoc's 8-12 images per class are too noisy for "
        "per-class claims; aggregating to ~5 categories gives enough images "
        "per category to support a comparison."
    ),
}

print(f"Aggregation level recorded:")
print(f"  choice = '{precommit['aggregation_level']['choice']}'")

### 8.2) Pre-commit your class-space remapping

PlantVillage has 38 classes; PlantDoc has 27. A direct comparison
isn't possible — your test has to either run on the intersection of
the two class spaces or on a remapping at the aggregation level you
just chose. This subsection is where you write that remapping.

If you chose **disease_category** in 8.1, the code cell below
provides a starter mapping covering all 38 PV classes. **Treat it
as a draft, not authoritative.** Each entry is a verifiable claim
("Late blight is fungal," "Citrus greening is bacterial") that you
should check against your reading list before locking it in. The
starter is based on common plant-pathology references, but plant
pathology has real disagreements about some categorizations:

- **Phytophthora** (Late blight on potato and tomato) is technically
  an oomycete, not a fungus. Most introductory references group it
  with fungi for convenience; some don't. Decide which side of that
  you're on.
- **Spider mites** are an arthropod pest, not a microbial disease.
  Categorized as `pest` in the starter so they don't get dropped,
  but you could legitimately choose to drop them, group them with
  healthy ("not a real disease"), or treat them on their own.
- **Citrus greening** (Huanglongbing) is caused by a *Liberibacter*
  bacterium spread by an insect vector. The starter calls it
  bacterial; an alternative ("vector-borne") would be defensible.

If you chose **host_group** in 8.1, replace the starter dict with
your own — the mapping is derivable from the `host` column of the
metadata DataFrame, since each `class_label` corresponds to a
single host. The code is short:
`{label: label.split("___")[0] for label in sorted(metadata["class_label"].unique())}`.

If you chose **your own grouping** in 8.1, write your own dict
from scratch. The starter dict below tells you which PV classes
you have to assign a label to.

Edit the starter dict (or replace it), then write your reasoning
in the code cell.

In [ ]:
### 8.3) Starter PV → PD class-space remapping
# Starter mapping at the host-group aggregation level.
# Keys are PV `class_label` strings (the host___disease folder-name format).
# Values are host group names that should align with PD's `host` column.
# Edit this dict to match your own aggregation level and your own
# remapping decisions before saving your precommit.json.
pv_to_aggregated = {
    "Apple___Apple_scab":                                 "Apple",
    "Apple___Black_rot":                                  "Apple",
    "Apple___Cedar_apple_rust":                           "Apple",
    "Apple___healthy":                                    "Apple",
    "Blueberry___healthy":                                "Blueberry",      # PV-only
    "Cherry_(including_sour)___Powdery_mildew":           "Cherry",         # PV-only
    "Cherry_(including_sour)___healthy":                  "Cherry",         # PV-only
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "Corn",
    "Corn_(maize)___Common_rust_":                        "Corn",
    "Corn_(maize)___Northern_Leaf_Blight":                "Corn",
    "Corn_(maize)___healthy":                             "Corn",
    "Grape___Black_rot":                                  "Grape",
    "Grape___Esca_(Black_Measles)":                       "Grape",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)":         "Grape",
    "Grape___healthy":                                    "Grape",
    "Orange___Haunglongbing_(Citrus_greening)":           "Orange",         # PV-only
    "Peach___Bacterial_spot":                             "Peach",          # PV-only
    "Peach___healthy":                                    "Peach",          # PV-only
    "Pepper,_bell___Bacterial_spot":                      "Bell pepper",    # PD spells it "Bell pepper"
    "Pepper,_bell___healthy":                             "Bell pepper",
    "Potato___Early_blight":                              "Potato",
    "Potato___Late_blight":                               "Potato",
    "Potato___healthy":                                   "Potato",
    "Raspberry___healthy":                                "Raspberry",      # PV-only
    "Soybean___healthy":                                  "Soybean",        # PD spells it "Soyabean"
    "Squash___Powdery_mildew":                            "Squash",         # PV-only
    "Strawberry___Leaf_scorch":                           "Strawberry",
    "Strawberry___healthy":                               "Strawberry",
    "Tomato___Bacterial_spot":                            "Tomato",
    "Tomato___Early_blight":                              "Tomato",
    "Tomato___Late_blight":                               "Tomato",
    "Tomato___Leaf_Mold":                                 "Tomato",
    "Tomato___Septoria_leaf_spot":                        "Tomato",
    "Tomato___Spider_mites Two-spotted_spider_mite":      "Tomato",
    "Tomato___Target_Spot":                               "Tomato",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":             "Tomato",
    "Tomato___Tomato_mosaic_virus":                       "Tomato",
    "Tomato___healthy":                                   "Tomato",
}

# PD's host column is already at this aggregation level — so the PD
# side of the remapping is just metadata['host'] directly (modulo the
# Soybean/Soyabean and Corn/Corn_(maize) spelling differences you saw
# in Section 4).
print(f"PV classes mapped:        {len(pv_to_aggregated)}")
print(f"Unique aggregated groups: {sorted(set(pv_to_aggregated.values()))}")
print(f"PD unique hosts:          {sorted(metadata['host'].unique())}")

In [ ]:
### 8.2) Record class-space remapping

# Starter mapping for the `disease_category` aggregation level.
# Edit this dict to reflect your decisions. Categories used:
# fungal, bacterial, viral, pest, healthy.
#
# Verify each row against your reading list before submitting.
starter_remapping = {
    "Apple___Apple_scab": "fungal",
    "Apple___Black_rot": "fungal",
    "Apple___Cedar_apple_rust": "fungal",
    "Apple___healthy": "healthy",
    "Blueberry___healthy": "healthy",
    "Cherry_(including_sour)___Powdery_mildew": "fungal",
    "Cherry_(including_sour)___healthy": "healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "fungal",
    "Corn_(maize)___Common_rust_": "fungal",
    "Corn_(maize)___Northern_Leaf_Blight": "fungal",
    "Corn_(maize)___healthy": "healthy",
    "Grape___Black_rot": "fungal",
    "Grape___Esca_(Black_Measles)": "fungal",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": "fungal",
    "Grape___healthy": "healthy",
    "Orange___Haunglongbing_(Citrus_greening)": "bacterial",
    "Peach___Bacterial_spot": "bacterial",
    "Peach___healthy": "healthy",
    "Pepper,_bell___Bacterial_spot": "bacterial",
    "Pepper,_bell___healthy": "healthy",
    "Potato___Early_blight": "fungal",
    "Potato___Late_blight": "fungal",
    "Potato___healthy": "healthy",
    "Raspberry___healthy": "healthy",
    "Soybean___healthy": "healthy",
    "Squash___Powdery_mildew": "fungal",
    "Strawberry___Leaf_scorch": "fungal",
    "Strawberry___healthy": "healthy",
    "Tomato___Bacterial_spot": "bacterial",
    "Tomato___Early_blight": "fungal",
    "Tomato___Late_blight": "fungal",
    "Tomato___Leaf_Mold": "fungal",
    "Tomato___Septoria_leaf_spot": "fungal",
    "Tomato___Spider_mites Two-spotted_spider_mite": "pest",
    "Tomato___Target_Spot": "fungal",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "viral",
    "Tomato___Tomato_mosaic_virus": "viral",
    "Tomato___healthy": "healthy",
}

precommit["class_space_remapping"] = {
    "scheme": "PlantVillage class_label -> disease category",
    "categories_used": sorted(set(starter_remapping.values())),
    "mapping": starter_remapping,
    "reasoning": (
        # Edit this string. Defend your category assignments, especially
        # the ones where references disagree (Phytophthora as fungal vs.
        # oomycete, spider mites as pest vs. drop, etc.). Name the sources
        # you used.
        "Five categories: fungal (the largest bucket; most PV diseases), "
        "bacterial (Citrus greening, Peach bacterial spot, Pepper and "
        "Tomato bacterial spot), viral (Tomato Yellow Leaf Curl Virus and "
        "Tomato mosaic virus), pest (spider mites — an arthropod, "
        "categorized separately rather than dropped), and healthy. Late "
        "blight (Phytophthora) is grouped with fungal here at "
        "student-level resolution; the formal taxonomy places it as an "
        "oomycete but introductory plant-pathology references commonly "
        "bundle it with fungi."
    ),
}

n_classes = len(starter_remapping)
n_categories = len(set(starter_remapping.values()))
print(f"Class-space remapping recorded:")
print(f"  classes mapped:    {n_classes}")
print(f"  categories used:   {n_categories}")
print(f"  categories:        {sorted(set(starter_remapping.values()))}")

### 8.3) Pre-commit your statistical test

The R2-Q1 hypothesis is that PV-internal accuracy and PV → PD
accuracy will be **statistically distinguishable** at your chosen
aggregation level. "Statistically distinguishable" is your call to
define before you measure — if you pick the test after seeing the
numbers, you'll end up choosing the test that returns the verdict
you expected, and your claim stops being a real test.

Three candidate tests, in decreasing simplicity:

- **Bootstrap confidence intervals on aggregate accuracy.** Resample
  your test set with replacement, compute aggregate accuracy on each
  resample, build a 95% CI around the mean. Do this for both PV
  internal and PV → PD. If the CIs don't overlap, the two
  accuracies are distinguishable. Conceptually clean; no
  distributional assumptions; easy to implement with NumPy.
- **Bootstrap comparison of paired distributions.** A more direct
  version: compute the difference between PV-internal and PV → PD
  on each bootstrap resample, build a CI around that difference,
  check whether zero falls inside. Stronger than the non-overlap
  check because it doesn't suffer from the "two CIs can fail to
  overlap even when their difference is significant" artifact.
- **A formal test of difference** (McNemar's test, paired t-test
  on per-class accuracies, etc.). More machinery, more assumptions,
  but produces a single p-value that's easier to report.

Pick one. The bootstrap-CI option is the most defensible default
for an introductory student project; the formal test is appropriate
if you have a specific reason to prefer it.

In [ ]:
### 8.3) Record statistical test
precommit["statistical_test"] = {
    "choice": "bootstrap_ci",       # edit to your choice
    "n_bootstrap": 1000,             # number of bootstrap resamples
    "alpha": 0.05,                   # 95% confidence level
    "reasoning": (
        # Edit this string. Defend your choice of test and your choice of
        # parameters (n_bootstrap, alpha). 2-4 sentences.
        "Bootstrap confidence intervals around aggregate accuracy on "
        "PV-internal vs PV->PD. Two intervals that don't overlap means "
        "the accuracies are distinguishable. Bootstrap is the right "
        "choice here because aggregate accuracy on ~5 categories doesn't "
        "have an obvious parametric form, and the bootstrap avoids the "
        "assumption-heavy alternatives. n=1000 resamples is plenty for "
        "stable CI estimates at this aggregation level."
    ),
}

print(f"Statistical test recorded:")
print(f"  choice = '{precommit['statistical_test']['choice']}'")
print(f"  alpha  = {precommit['statistical_test']['alpha']}")

### 8.4) Write the file

The `precommit` dictionary now has all three of your decisions. The
cell below writes it to `precommit.json` in your R2-Q1 output
directory using `json.dump`. The file is your **EQ#1 deliverable**.

`indent=2` makes the file readable when you (or a mentor) open it
in a text editor. JSON's key order matches Python's dict order, so
the three decisions appear in the same sequence you wrote them.

In [ ]:
### 8.4) Save precommit.json
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path, "w") as f:
    json.dump(precommit, f, indent=2)

print(f"Wrote: {precommit_path}")
print(f"  size: {precommit_path.stat().st_size:,} bytes")
print(f"  keys: {list(precommit.keys())}")

### 8.5) Sanity-load

The cell below reads `precommit.json` back from disk and compares
it to the dictionary you just wrote. The round-trip catches any
typos, JSON serialization edge cases, or accidental overwrites
before Notebook 03 picks the file up next week.

In [ ]:
### 8.5) Read back and verify
with open(precommit_path) as f:
    reloaded = json.load(f)

# Round-trip: the reloaded dict should match what was written.
assert reloaded == precommit, "Reloaded precommit differs from in-memory"

# Confirm all three decisions are present by name.
expected_keys = {"aggregation_level", "class_space_remapping", "statistical_test"}
assert set(reloaded.keys()) == expected_keys, (
    f"Expected keys {expected_keys}, got {set(reloaded.keys())}"
)

print(f"Round-trip verified: 3 of 3 decisions saved.")
for key in reloaded:
    print(f"  - {key}")

**You're done with Week 1 of R2-Q1.**

You have one file saved to your R2-Q1 output directory:

- `precommit.json` — your three Week 1 decisions and the reasoning
  for each. This is your EQ#1 deliverable.

**Submit EQ#1 by the end of Week 1.** Your EQ#1 writeup draws on
the `reasoning` fields plus whatever additional context the
deliverables page asks for: the foundational claim, your
understanding of the PV/PD class-space mismatch, your defense of
each pre-commit decision.

Week 2 picks up in Notebook 03 (`03_baseline_classifier.ipynb`),
which trains a baseline CNN classifier on PlantVillage and
evaluates its PV-internal accuracy. That accuracy becomes the
baseline you'll compare PV → PD accuracy against in Week 3.
Notebook 03 doesn't open `precommit.json` itself — it just confirms
the file is in your output directory and lets you continue. Week 3
is where the contents of this file actually get used.

## 9) What's next

You've now seen PlantDoc's structure (28 classes, 13 hosts, lowercased
disease vocabulary), its irregularities (inconsistent naming, the
orphan class), and its visual character (heterogeneous field-condition
photographs). You've also compared it directly with PlantVillage and
committed to the three methodology decisions that make your transfer
evaluation rigorous.

Notebook 03 (`03_baseline_classifier.ipynb`) trains a baseline CNN
classifier on PlantVillage. It expects your `precommit.json` to exist
in your R2-Q1 output directory — but it won't read the file itself.
The contents of your pre-commits start to matter in Week 3, when
Notebook 04 applies the trained classifier to PlantDoc using your
remapping. So write your reasoning fields with Week 3 in mind, not
Week 2. Don't move on to Notebook 03 until your pre-commits are saved.

This notebook produced one saved artifact (`precommit.json`, written
by you). See you in Week 2.